# 01 — Sentinel-2 Point-based Land Cover Classification

## การจำแนกสิ่งปกคลุมดินด้วย Sentinel-2 และ Random Forest
**Google Earth Engine Python API + Google Colab**

**พื้นที่ศึกษา:** อำเภอเมืองพิษณุโลก จังหวัดพิษณุโลก  
**ช่วงข้อมูล:** มกราคม 2021  
**ระดับ:** ปริญญาตรีปี 3–4 / ปริญญาโทด้านสิ่งแวดล้อม

| Class ID | Class |
|---:|---|
| 0 | Urban / Built-up |
| 1 | Water |
| 2 | Forest / Tree cover |
| 3 | Agriculture |
| 4 | Bare soil / Open land |

> Notebook นี้ใช้ **point-based supervised classification** ก่อน  
> เรื่อง rectangle/polygon training จะเรียนใน Notebook 02

## Learning objectives

เมื่อจบ Notebook นี้ ผู้เรียนควรสามารถ

1. อธิบาย True Color และ False Color ของ Sentinel-2
2. ใช้ spectral information ตรวจสอบ reference points
3. คำนวณ NDVI, NDWI และ NDBI
4. อธิบาย predictor, training sample และ validation sample
5. จำแนกสิ่งปกคลุมดินด้วย Random Forest
6. อ่าน Confusion Matrix, OA, PA, UA และ F1
7. ส่งออกผลจำแนกเป็น GeoTIFF เพื่อนำไปใช้ต่อใน QGIS

## Workflow

```text
Sentinel-2 Surface Reflectance
            ↓
True Color / False Color
            ↓
Spectral bands + indices
            ↓
Reference points
            ↓
Fixed stratified split
      ┌─────┴─────┐
      ↓           ↓
  Training     Validation
      ↓           │
Random Forest     │
      ↓           │
Classification    │
      └─────┬─────┘
            ↓
    Accuracy Assessment
            ↓
      GeoTIFF → QGIS
```

**แนวคิดสำคัญ:** Classification quality ขึ้นอยู่กับ image quality, predictor design, reference data และ sampling design ไม่ใช่ algorithm เพียงอย่างเดียว

In [1]:
import ee
import folium
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display

print("Earth Engine API version:", ee.__version__)
print("Folium version:", folium.__version__)

Earth Engine API version: 1.7.39
Folium version: 0.20.0


In [3]:
# เปลี่ยนเป็น Google Cloud Project ID ของนิสิตเอง
PROJECT = "teach-gee-github-lulc"

ee.Authenticate()
ee.Initialize(project=PROJECT)

print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## Folium + Earth Engine

Function ต่อไปนี้ใช้เชื่อม Earth Engine map tiles กับ Folium

**ไม่จำเป็นต้องจำ syntax ภายใน function** — ให้เข้าใจเพียงว่า Earth Engine ประมวลผลบน server และ Folium ใช้แสดงผลแบบ interactive map

In [4]:
def add_ee_layer(map_object, ee_image, vis_params, name,
                 shown=True, opacity=1.0):
    map_id = ee.Image(ee_image).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
        show=shown,
        opacity=opacity,
    ).add_to(map_object)


def add_esri_imagery(map_object, shown=False):
    folium.TileLayer(
        tiles=(
            "https://server.arcgisonline.com/ArcGIS/rest/services/"
            "World_Imagery/MapServer/tile/{z}/{y}/{x}"
        ),
        attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
        name="Esri World Imagery",
        overlay=False,
        control=True,
        show=shown,
    ).add_to(map_object)


CLASS_COLORS = {
    0: "#d73027",
    1: "#2166ac",
    2: "#1a9850",
    3: "#fee08b",
    4: "#a6611a",
}

CLASS_NAMES = {
    0: "Urban",
    1: "Water",
    2: "Forest",
    3: "Agriculture",
    4: "Bare soil",
}

CLASS_PALETTE = [CLASS_COLORS[i].replace("#", "") for i in range(5)]

# Part 1 — Study area

ใช้จุดอ้างอิงเดิมใกล้ตัวเมืองพิษณุโลก

```text
Longitude ≈ 100.26575°E
Latitude  ≈ 16.82277°N
```

จากนั้นใช้ **point-in-polygon** เลือกหน่วยปกครองระดับอำเภอจาก FAO GAUL 2025 Level 2

> Administrative boundary เป็นขอบเขตการปกครอง ไม่ใช่ physical urban extent โดยอัตโนมัติ

In [5]:
roi = ee.Geometry.Point([100.26575, 16.82277])

gaul2 = (
    ee.FeatureCollection("FAO/GAUL/2025/level2")
    .filter(ee.Filter.eq("GAUL0_NAME", "Thailand"))
)

district_fc = gaul2.filterBounds(roi)
study_area = district_fc.geometry()

print("District candidate(s):")
display(district_fc.aggregate_array("DISP_EN").getInfo())

District candidate(s):


['Mueang Phitsanulok, Phitsanulok, Thailand']

In [6]:
Map = folium.Map(
    location=[16.82, 100.27],
    zoom_start=10,
    tiles=None,
)

add_esri_imagery(Map, shown=True)

boundary_image = district_fc.style(
    color="FFFFFF",
    fillColor="00000000",
    width=2,
)

add_ee_layer(
    Map,
    boundary_image,
    {},
    "Study area boundary",
    shown=True,
)

folium.Marker(
    location=[16.82277, 100.26575],
    tooltip="ROI reference point",
).add_to(Map)

folium.LayerControl(collapsed=False).add_to(Map)
Map

# Part 2 — Sentinel-2 Surface Reflectance

ใช้ `COPERNICUS/S2_SR_HARMONIZED`

| Band | Spectral region | Native resolution | บทบาทเบื้องต้น |
|---|---|---:|---|
| B2 | Blue | 10 m | visible / water |
| B3 | Green | 10 m | vegetation / water |
| B4 | Red | 10 m | chlorophyll absorption |
| B8 | NIR | 10 m | vegetation |
| B11 | SWIR1 | 20 m | moisture / soil / built-up |
| B12 | SWIR2 | 20 m | soil / built-up / moisture |

Surface Reflectance เก็บเป็น scaled integer

$$
\rho = DN \times 0.0001
$$

- $DN$ = stored pixel value
- $\rho$ = Surface Reflectance

> การทำงานที่ scale 10 m ไม่ได้สร้างรายละเอียดใหม่ให้ B11/B12 ซึ่งมี native resolution 20 m

In [7]:
START_DATE = "2021-01-01"
END_DATE = "2021-02-01"

SPECTRAL_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]

s2_raw = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(study_area)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 50))
)

print("Sentinel-2 scenes after scene-level filtering:")
display(s2_raw.size().getInfo())

Sentinel-2 scenes after scene-level filtering:


14

## Scene cloud percentage ≠ pixel-level cloud mask

`CLOUDY_PIXEL_PERCENTAGE` ใช้คัดภาพทั้ง scene

แต่ภาพที่ผ่านเกณฑ์อาจยังมีเมฆหรือเงาเมฆอยู่ จึงใช้ **SCL — Scene Classification Layer** mask pixel ที่ไม่เหมาะสม แล้วจึงสร้าง median composite

> Median composite ช่วยลดค่าผิดปกติ แต่ไม่ได้หมายความว่า cloud-free โดยอัตโนมัติ

In [8]:
def mask_s2_scl(image):
    scl = image.select("SCL")

    mask = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    reflectance = (
        image
        .updateMask(mask)
        .select(SPECTRAL_BANDS)
        .multiply(0.0001)
    )

    return reflectance.copyProperties(
        image,
        ["system:time_start"]
    )


s2_masked = s2_raw.map(mask_s2_scl)

s2 = (
    s2_masked
    .median()
    .clip(study_area)
)

print("Monthly median Sentinel-2 composite created.")

Monthly median Sentinel-2 composite created.


# Part 3 — Visual interpretation before classification

### True Color

$$
R=B4,\qquad G=B3,\qquad B=B2
$$

### False Color — Vegetation

$$
R=B8,\qquad G=B4,\qquad B=B3
$$

### False Color — Urban / Bare soil

$$
R=B12,\qquad G=B11,\qquad B=B4
$$

### False Color — Agriculture / moisture contrast

$$
R=B11,\qquad G=B8,\qquad B=B2
$$

**อย่าเลือกหรือยืนยัน training sample จาก True Color เพียงอย่างเดียว**

In [9]:
true_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0.02,
    "max": 0.30,
    "gamma": 1.1,
}

veg_vis = {
    "bands": ["B8", "B4", "B3"],
    "min": 0.02,
    "max": 0.35,
    "gamma": 1.1,
}

urban_vis = {
    "bands": ["B12", "B11", "B4"],
    "min": 0.02,
    "max": 0.35,
    "gamma": 1.1,
}

agri_vis = {
    "bands": ["B11", "B8", "B2"],
    "min": 0.02,
    "max": 0.35,
    "gamma": 1.1,
}

Map = folium.Map(
    location=[16.82, 100.27],
    zoom_start=10,
    tiles=None,
)

add_esri_imagery(Map, shown=False)
add_ee_layer(Map, s2, true_vis, "S2 True Color", shown=True)
add_ee_layer(Map, s2, veg_vis, "S2 False Color — Vegetation", shown=False)
add_ee_layer(Map, s2, urban_vis, "S2 False Color — Urban/Bare", shown=False)
add_ee_layer(Map, s2, agri_vis, "S2 False Color — Agriculture", shown=False)
add_ee_layer(Map, boundary_image, {}, "Study area boundary", shown=True)

folium.LayerControl(collapsed=False).add_to(Map)
Map

## Mini activity — อ่านภาพก่อนใช้ Machine Learning

1. Urban กับ Bare soil แยกจากกันชัดใน True Color หรือไม่?
2. False Color แบบใดทำให้ vegetation เด่น?
3. พื้นที่เกษตรทุกแปลงมีสีเหมือนกันหรือไม่?
4. น้ำ ตลิ่ง และดินชื้นมีโอกาสสับสนกันหรือไม่?
5. เหตุใดแผนที่ฐานความละเอียดสูงจึงเหมาะสำหรับช่วยตรวจ reference data แต่ไม่ควรถูกนำมาเป็น predictor โดยไม่มีหลักการ?

> Machine Learning ไม่สามารถแก้ reference data ที่ติดป้าย class ผิดได้

# Part 4 — Spectral indices

### NDVI

$$
NDVI = \frac{NIR-Red}{NIR+Red}
     = \frac{B8-B4}{B8+B4}
$$

### NDWI

$$
NDWI = \frac{Green-NIR}{Green+NIR}
     = \frac{B3-B8}{B3+B8}
$$

### NDBI

$$
NDBI = \frac{SWIR1-NIR}{SWIR1+NIR}
     = \frac{B11-B8}{B11+B8}
$$

> NDVI ไม่ใช่ Forest class  
> NDWI ไม่ใช่ Water class  
> NDBI ไม่ใช่ Urban class  
>
> ทั้งหมดเป็น **predictor variables**

In [10]:
ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndwi = s2.normalizedDifference(["B3", "B8"]).rename("NDWI")
ndbi = s2.normalizedDifference(["B11", "B8"]).rename("NDBI")

indices = ndvi.addBands([ndwi, ndbi])

print("Index bands:")
display(indices.bandNames().getInfo())

Index bands:


['NDVI', 'NDWI', 'NDBI']

In [11]:
Map = folium.Map(
    location=[16.82, 100.27],
    zoom_start=10,
    tiles=None,
)

add_esri_imagery(Map, shown=False)
add_ee_layer(Map, s2, true_vis, "S2 True Color", shown=True)
add_ee_layer(Map, ndvi, {"min": -0.2, "max": 0.8}, "NDVI", shown=False)
add_ee_layer(Map, ndwi, {"min": -0.6, "max": 0.6}, "NDWI", shown=False)
add_ee_layer(Map, ndbi, {"min": -0.5, "max": 0.5}, "NDBI", shown=False)

folium.LayerControl(collapsed=False).add_to(Map)
Map

## Predictor / Feature vector

สำหรับแต่ละตำแหน่ง $i$

$$
\mathbf{x}_i =
[
B2_i,\,
B3_i,\,
B4_i,\,
B8_i,\,
B11_i,\,
B12_i,\,
NDVI_i,\,
NDWI_i,\,
NDBI_i
]
$$

และ class label

$$
y_i \in \{0,1,2,3,4\}
$$

Classifier ไม่ได้มองภาพเหมือนมนุษย์ แต่เรียนรู้จาก **ค่าตัวเลขหลายตัวของแต่ละตัวอย่าง**

In [12]:
PREDICTOR_BANDS = [
    "B2", "B3", "B4", "B8", "B11", "B12",
    "NDVI", "NDWI", "NDBI"
]

predictors = (
    s2
    .addBands(indices)
    .select(PREDICTOR_BANDS)
)

print("Predictor bands:")
display(predictors.bandNames().getInfo())

Predictor bands:


['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI']

# Part 5 — Reference points

ข้อมูลเดิมมีทั้งหมด **101 จุด**

| Class | จำนวน |
|---|---:|
| Urban | 20 |
| Water | 20 |
| Forest | 20 |
| Agriculture | 20 |
| Bare soil | 21 |

ใช้ dataset เดียวกันต่อเนื่องใน Notebook 01–05

> `Agriculture` คือพื้นที่เกษตรทั่วไป **ไม่เท่ากับ Rice โดยอัตโนมัติ**

In [13]:
# Recommended repository:
#
# Teach-GEE-LULC/
# ├── 01_Sentinel2_Point_LULC_Classification.ipynb
# └── data/
#     └── phitsanulok_lulc_points.csv

LOCAL_CSV = Path("/content/phitsanulok_lulc_points.csv")

GITHUB_CSV = (
    "https://raw.githubusercontent.com/"
    "nattaponm/Teach-GEE-LULC/main/"
    "data/phitsanulok_lulc_points.csv"
)

if LOCAL_CSV.exists():
    samples_df = pd.read_csv(LOCAL_CSV)
    print("Loaded:", LOCAL_CSV)
else:
    try:
        samples_df = pd.read_csv(GITHUB_CSV)
        print("Loaded from GitHub:", GITHUB_CSV)
    except Exception:
        print("CSV is not available on GitHub yet. Please upload it to Colab.")
        from google.colab import files

        uploaded = files.upload()
        filename = next(iter(uploaded))
        samples_df = pd.read_csv(filename)

display(samples_df.head())
print("Number of reference points:", len(samples_df))

Loaded from GitHub: https://raw.githubusercontent.com/nattaponm/Teach-GEE-LULC/main/data/phitsanulok_lulc_points.csv


,sample_id,longitude,latitude,class_id,class_name,split
0,UR001,100.269879,16.817238,0,Urban,train
1,UR002,100.285468,16.817505,0,Urban,validation
2,UR003,100.256249,16.804334,0,Urban,validation
3,UR004,100.279540,16.782361,0,Urban,train
4,UR005,100.197563,16.753218,0,Urban,validation


Number of reference points: 101


## Training vs Validation

ใช้ **stratified hold-out validation**

```text
Reference samples
       ↓
split once with fixed seed
       ↓
┌────────────────┬────────────────┐
│ Training       │ Validation     │
│ ~70%           │ ~30%           │
└────────────────┴────────────────┘
```

- Training = model ใช้เรียนรู้
- Validation = กันออกไว้ก่อน train

เราใช้ split เดิมใน Notebook 01–05 เพื่อให้เปรียบเทียบ S2, S1 และ S1+S2 ได้อย่างยุติธรรม

> นี่เป็น **internal hold-out validation** ไม่ใช่ independent probability-based accuracy assessment สำหรับงานผลิตแผนที่มาตรฐาน

In [14]:
SEED = 42
TRAIN_FRACTION = 0.70

def make_stratified_split(df, train_fraction=0.70, seed=42):
    out = df.copy()
    out["split"] = "validation"

    rng = np.random.default_rng(seed)

    for class_id, indices in out.groupby("class_id").groups.items():
        indices = np.array(list(indices))
        rng.shuffle(indices)
        n_train = int(round(len(indices) * train_fraction))
        out.loc[indices[:n_train], "split"] = "train"

    return out


if "split" not in samples_df.columns:
    samples_df = make_stratified_split(
        samples_df,
        train_fraction=TRAIN_FRACTION,
        seed=SEED,
    )
    print("Created fixed stratified split.")
else:
    print("Using the fixed split stored in the CSV.")

samples_df["class_id"] = samples_df["class_id"].astype(int)

display(
    pd.crosstab(
        samples_df["class_name"],
        samples_df["split"],
        margins=True,
    )
)

Using the fixed split stored in the CSV.


split,train,validation,All
class_name,,,
Agriculture,14,6,20
Bare soil,15,6,21
Forest,14,6,20
Urban,14,6,20
Water,14,6,20
All,71,30,101


In [15]:
def dataframe_to_ee_points(df):
    features = []

    for row in df.itertuples(index=False):
        geometry = ee.Geometry.Point(
            [float(row.longitude), float(row.latitude)]
        )

        properties = {
            "sample_id": str(row.sample_id),
            "longitude": float(row.longitude),
            "latitude": float(row.latitude),
            "class_id": int(row.class_id),
            "class_name": str(row.class_name),
            "split": str(row.split),
        }

        features.append(
            ee.Feature(geometry, properties)
        )

    return ee.FeatureCollection(features)


all_points = dataframe_to_ee_points(samples_df)

training_points = all_points.filter(
    ee.Filter.eq("split", "train")
)

validation_points = all_points.filter(
    ee.Filter.eq("split", "validation")
)

print("Training locations:", training_points.size().getInfo())
print("Validation locations:", validation_points.size().getInfo())

Training locations: 71
Validation locations: 30


In [16]:
Map = folium.Map(
    location=[16.82, 100.27],
    zoom_start=10,
    tiles=None,
)

add_esri_imagery(Map, shown=True)
add_ee_layer(Map, s2, true_vis, "S2 True Color", shown=False)
add_ee_layer(Map, s2, veg_vis, "S2 False Color — Vegetation", shown=False)
add_ee_layer(Map, s2, urban_vis, "S2 False Color — Urban/Bare", shown=False)

training_group = folium.FeatureGroup(name="Training points", show=True)
validation_group = folium.FeatureGroup(name="Validation points", show=True)

for row in samples_df.itertuples(index=False):
    color = CLASS_COLORS[int(row.class_id)]
    popup = (
        f"{row.sample_id}<br>"
        f"Class: {row.class_name}<br>"
        f"Split: {row.split}"
    )

    if row.split == "train":
        folium.CircleMarker(
            location=[row.latitude, row.longitude],
            radius=4,
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            tooltip=popup,
        ).add_to(training_group)
    else:
        folium.CircleMarker(
            location=[row.latitude, row.longitude],
            radius=6,
            color="black",
            weight=2,
            fill=False,
            tooltip=popup,
        ).add_to(validation_group)

training_group.add_to(Map)
validation_group.add_to(Map)

folium.LayerControl(collapsed=False).add_to(Map)
Map

## Spatial QC ก่อน train

1. จุดแต่ละ class กระจายครอบคลุมความหลากหลายหรือไม่?
2. มีจุดใดอยู่ใกล้ขอบเขตของสอง classes หรือไม่?
3. Validation points กระจุกตัวหรือไม่?
4. Water อยู่กลางน้ำหรือใกล้ตลิ่ง?
5. Urban มีโอกาสเป็น bare soil/open land หรือไม่?

**Reference-data quality เป็นส่วนหนึ่งของ model quality**

# Part 6 — Random Forest

Random Forest ใช้ decision trees หลายต้นแล้วตัดสินด้วย majority vote

$$
\hat{y}
=
\operatorname{mode}
\left[
h_1(\mathbf{x}),
h_2(\mathbf{x}),
\dots,
h_B(\mathbf{x})
\right]
$$

- $h_b(\mathbf{x})$ = prediction จาก tree ที่ $b$
- $B$ = จำนวน trees
- $\hat{y}$ = predicted class

Notebook นี้ใช้ **100 trees** และยังไม่ทำ hyperparameter tuning

In [17]:
training_data = predictors.sampleRegions(
    collection=training_points,
    properties=["sample_id", "class_id", "class_name"],
    scale=10,
    geometries=True,
)

print("Training locations supplied:", training_points.size().getInfo())
print("Training samples retained after image masking:", training_data.size().getInfo())

display(
    training_data.aggregate_histogram("class_id").getInfo()
)

Training locations supplied: 71
Training samples retained after image masking: 64


{'0': 14, '1': 14, '2': 7, '3': 14, '4': 15}

In [18]:
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=SEED,
).train(
    features=training_data,
    classProperty="class_id",
    inputProperties=PREDICTOR_BANDS,
)

print("Random Forest trained.")

Random Forest trained.


In [19]:
classified = (
    predictors
    .classify(classifier)
    .rename("landcover")
    .clip(study_area)
)

print("Classification image created.")

Classification image created.


In [20]:
class_vis = {
    "min": 0,
    "max": 4,
    "palette": CLASS_PALETTE,
}

Map = folium.Map(
    location=[16.82, 100.27],
    zoom_start=10,
    tiles=None,
)

add_esri_imagery(Map, shown=False)
add_ee_layer(Map, s2, true_vis, "S2 True Color", shown=True)
add_ee_layer(Map, classified, class_vis, "S2 RF Classification", shown=True)

legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background-color:white;padding:10px;border:1px solid grey;font-size:13px;">'
    '<b>LULC classes</b><br>'
    '<span style="color:#d73027;">●</span> 0 Urban<br>'
    '<span style="color:#2166ac;">●</span> 1 Water<br>'
    '<span style="color:#1a9850;">●</span> 2 Forest<br>'
    '<span style="color:#fee08b;">●</span> 3 Agriculture<br>'
    '<span style="color:#a6611a;">●</span> 4 Bare soil'
    '</div>'
)

Map.get_root().html.add_child(
    folium.Element(legend_html)
)

folium.LayerControl(collapsed=False).add_to(Map)
Map

# Part 7 — Accuracy Assessment

ให้ Confusion Matrix เป็น

$$
\mathbf{M}=[n_{ij}]
$$

- row $i$ = reference / actual
- column $j$ = predicted
- diagonal $n_{ii}$ = correctly classified samples

### Overall Accuracy

$$
OA=
\frac{\sum_i n_{ii}}
{\sum_i\sum_j n_{ij}}
$$

### Producer's Accuracy

$$
PA_i=
\frac{n_{ii}}
{\sum_j n_{ij}}
$$

$$
OE_i=1-PA_i
$$

### User's Accuracy

$$
UA_i=
\frac{n_{ii}}
{\sum_j n_{ji}}
$$

$$
CE_i=1-UA_i
$$

### F1

$$
F1_i=
2
\frac{PA_iUA_i}
{PA_i+UA_i}
$$

> Overall Accuracy สูง ไม่ได้หมายความว่าทุก class ดี

In [21]:
train_cm = classifier.confusionMatrix()

print("Training / resubstitution confusion matrix:")
display(train_cm.getInfo())

print(
    "Training overall accuracy:",
    round(train_cm.accuracy().getInfo(), 4),
)

Training / resubstitution confusion matrix:


[[14, 0, 0, 0, 0],
 [0, 14, 0, 0, 0],
 [0, 0, 7, 0, 0],
 [0, 0, 0, 14, 0],
 [0, 0, 0, 0, 15]]

Training overall accuracy: 1


In [22]:
validation_data = predictors.sampleRegions(
    collection=validation_points,
    properties=["sample_id", "class_id", "class_name"],
    scale=10,
    geometries=True,
)

validation_predictions = validation_data.classify(
    classifier,
    "predicted",
)

print("Validation locations supplied:", validation_points.size().getInfo())
print("Validation samples retained after image masking:", validation_data.size().getInfo())

Validation locations supplied: 30
Validation samples retained after image masking: 27


In [23]:
CLASS_ORDER = [0, 1, 2, 3, 4]

validation_cm = validation_predictions.errorMatrix(
    "class_id",
    "predicted",
    CLASS_ORDER,
)

cm_array = np.array(
    validation_cm.array().getInfo(),
    dtype=int,
)

cm_df = pd.DataFrame(
    cm_array,
    index=[CLASS_NAMES[i] for i in CLASS_ORDER],
    columns=[CLASS_NAMES[i] for i in CLASS_ORDER],
)

cm_df.index.name = "Reference / Actual"
cm_df.columns.name = "Predicted"

display(cm_df)

overall_accuracy = (
    np.trace(cm_array) / cm_array.sum()
    if cm_array.sum() > 0
    else np.nan
)

print("Validation Overall Accuracy:", round(overall_accuracy, 4))

Predicted,Urban,Water,Forest,Agriculture,Bare soil
Reference / Actual,,,,,
Urban,6,0,0,0,0
Water,0,6,0,0,0
Forest,0,0,2,1,0
Agriculture,0,0,1,5,0
Bare soil,0,0,0,0,6


Validation Overall Accuracy: 0.9259


In [24]:
diag = np.diag(cm_array).astype(float)
row_sum = cm_array.sum(axis=1).astype(float)
col_sum = cm_array.sum(axis=0).astype(float)

pa = np.divide(
    diag,
    row_sum,
    out=np.full_like(diag, np.nan),
    where=row_sum != 0,
)

ua = np.divide(
    diag,
    col_sum,
    out=np.full_like(diag, np.nan),
    where=col_sum != 0,
)

f1 = np.divide(
    2 * pa * ua,
    pa + ua,
    out=np.full_like(diag, np.nan),
    where=(pa + ua) != 0,
)

accuracy_by_class = pd.DataFrame({
    "class_id": CLASS_ORDER,
    "class_name": [CLASS_NAMES[i] for i in CLASS_ORDER],
    "Producer_Accuracy": pa,
    "User_Accuracy": ua,
    "F1": f1,
    "Validation_n": row_sum.astype(int),
})

display(
    accuracy_by_class.style.format({
        "Producer_Accuracy": "{:.3f}",
        "User_Accuracy": "{:.3f}",
        "F1": "{:.3f}",
    })
)

print("Overall Accuracy:", round(overall_accuracy, 3))

,class_id,class_name,Producer_Accuracy,User_Accuracy,F1,Validation_n
0,0,Urban,1.000,1.000,1.000,6
1,1,Water,1.000,1.000,1.000,6
2,2,Forest,0.667,0.667,0.667,3
3,3,Agriculture,0.833,0.833,0.833,6
4,4,Bare soil,1.000,1.000,1.000,6


Overall Accuracy: 0.926


## Interpretation questions

1. Class ใดจำแนกได้ดีที่สุด?
2. Urban สับสนกับ Bare soil หรือไม่? เพราะเหตุใดในเชิง spectral?
3. Forest และ Agriculture มี confusion กันหรือไม่?
4. Training accuracy สูงกว่า validation accuracy หรือไม่?
5. Validation ประมาณ 6 จุดต่อ class ทำให้ accuracy มีความไม่แน่นอนอย่างไร?
6. จุดที่ผิดอยู่ใกล้ class boundary หรือไม่?
7. ถ้าเพิ่ม training points แต่เลือก label ผิด ผลจะดีขึ้นจริงหรือไม่?

### Limitations

- reference samples มีจำนวนไม่มาก
- validation เป็น hold-out จาก dataset เดียวกับ training
- จุดอาจมี spatial clustering
- B11/B12 มี native resolution 20 m
- land cover เปลี่ยนตามเวลาได้
- map accuracy ไม่ได้แปลว่า map ถูกต้องทุกตำแหน่ง

# Part 8 — Export GeoTIFF for QGIS

Class values

```text
0 = Urban
1 = Water
2 = Forest
3 = Agriculture
4 = Bare soil
255 = NoData
```

Export เป็น GeoTIFF ไป Google Drive

```text
CRS   = EPSG:32647
Scale = 10 m
```

EPSG:32647 = WGS 84 / UTM Zone 47N

> `scale=10` คือ output grid ไม่ใช่การสร้างรายละเอียดใหม่ให้ B11/B12

In [25]:
NO_DATA = 255

classified_export = (
    classified
    .toByte()
    .unmask(NO_DATA)
)

export_task = ee.batch.Export.image.toDrive(
    image=classified_export,
    description="PHS_S2_Point_RF_LULC_2021",
    folder="GEE_LULC",
    fileNamePrefix="PHS_S2_Point_RF_LULC_2021",
    region=study_area,
    scale=10,
    crs="EPSG:32647",
    maxPixels=1e13,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True,
        "noData": NO_DATA,
    },
)

export_task.start()

print("GeoTIFF export task started.")
print(export_task.status())

GeoTIFF export task started.
{'state': 'READY', 'description': 'PHS_S2_Point_RF_LULC_2021', 'priority': 100, 'creation_timestamp_ms': 1791263329203, 'update_timestamp_ms': 1791263329203, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': '2OOG5ZTHA2KBIV2WP4MLWID2', 'name': 'projects/teach-gee-github-lulc/operations/2OOG5ZTHA2KBIV2WP4MLWID2'}


In [26]:
sample_export_task = ee.batch.Export.table.toDrive(
    collection=all_points,
    description="PHS_LULC_Reference_Points_FixedSplit",
    folder="GEE_LULC",
    fileNamePrefix="phitsanulok_lulc_points_split",
    fileFormat="CSV",
    selectors=[
        "sample_id",
        "longitude",
        "latitude",
        "class_id",
        "class_name",
        "split",
    ],
)

sample_export_task.start()

print("Reference CSV export task started.")
print(sample_export_task.status())

Reference CSV export task started.
{'state': 'READY', 'description': 'PHS_LULC_Reference_Points_FixedSplit', 'priority': 100, 'creation_timestamp_ms': 1791263331039, 'update_timestamp_ms': 1791263331039, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'MO22GALSGNO5ZZWVLMAYM734', 'name': 'projects/teach-gee-github-lulc/operations/MO22GALSGNO5ZZWVLMAYM734'}


# Exercise A — ตรวจจุดที่จำแนกผิด

เลือก validation points ที่จำแนกผิดอย่างน้อย 3 จุด แล้วตรวจด้วย

- True Color
- False Color — Vegetation
- False Color — Urban/Bare soil
- NDVI
- NDWI
- NDBI

อภิปรายว่าเหตุใด model จึงอาจจำแนกผิด

# Exercise B — Predictor experiment

เปรียบเทียบ

### Model A

```text
B2, B3, B4, B8, B11, B12
```

### Model B

```text
B2, B3, B4, B8, B11, B12,
NDVI, NDWI, NDBI
```

ควบคุมให้เหมือนกัน

- training points
- validation points
- Random Forest 100 trees

แล้วเปรียบเทียบ OA, PA, UA และ F1

**คำถาม:** การเพิ่ม indices ทำให้ทุก class ดีขึ้นหรือไม่?

# Take-home messages

```text
Satellite image
      +
Reference data
      +
Predictor design
      +
Classification model
      +
Hold-out validation
      =
Defensible classification workflow
```

1. Interpret image ก่อนตรวจ training data
2. False Color ช่วยแยก land cover บางชนิดได้ดีกว่า True Color
3. Spectral indices เป็น predictors ไม่ใช่ classes
4. Random Forest เรียนรู้จาก feature vectors + labels
5. Training accuracy ≠ validation accuracy
6. OA ต้องอ่านร่วมกับ class-specific accuracy
7. ต้องตรวจแผนที่กลับกับ imagery และ geographic context
8. GeoTIFF เชื่อม Earth Engine workflow ไป QGIS

## Next

**Notebook 02 — Sentinel-2 Rectangle-based LULC Classification**

ใช้ training locations เดิม แต่เปลี่ยนจาก **point → 20 × 20 m training window**  
และคง validation points ชุดเดิม

# References

- Sentinel-2 SR Harmonized  
  https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED

- FAO GAUL 2025 Level 2  
  https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2025_level2

- Random Forest  
  https://developers.google.com/earth-engine/apidocs/ee-classifier-smilerandomforest

- Error Matrix  
  https://developers.google.com/earth-engine/apidocs/ee-featurecollection-errormatrix

- Export image to Google Drive  
  https://developers.google.com/earth-engine/apidocs/export-image-todrive

### Teaching note

Notebook นี้พัฒนาต่อยอดจาก supervised-classification workflow เดิมของผู้สอน  
โดยปรับจาก JavaScript API เป็น Earth Engine Python API และใช้ reference dataset เดียวกันต่อเนื่องใน Notebook 01–05